### Imports

In [1]:
import os
from pathlib import Path
import json
import cv2
import supervision as sv
from tqdm import tqdm
import sys
import numpy as np
from collections import OrderedDict
from collections import defaultdict


### Initialization

In [2]:
sys.path.insert(0, "../../")
from config import MEDIA_PATH, TEMP_PATH, CROPPED_PATH

sys.path.insert(0, "../../packages/python")
from models import cell_segmentation as segmentators

IMG_TARGET_SIDE = 200

JSON_PATH = os.path.join(TEMP_PATH, 'datasets_area_data.json')

GENERATION_DATASET_PATH = os.path.join(CROPPED_PATH, 'generation')

SEED = 42


2025-09-27 09:02:35.226819: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 09:02:35.249356: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-27 09:02:35.850112: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Functions

In [3]:
def crop_and_save_detections(images_dir: Path, annotations_path: Path, output_dir: Path, resize_factor, area_data) -> None:
    """
    Loads COCO annotations, crops the detected objects from images, and saves
    them into class-specific folders.

    Args:
        images_dir (Path): The path to the directory containing the images.
        annotations_path (Path): The path to the COCO JSON annotation file.
        output_dir (Path): The path to the directory where cropped images will be saved.
    """
    # --- Validations ---
    if not images_dir.is_dir():
        print(f"Images dir not found: {images_dir}")
        return
    if not annotations_path.is_file():
        print(f"Annotations file not found: {annotations_path}")
        return
    
    # Ensure the main output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the dataset using supervision
    print("Loading dataset...")
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=str(images_dir),
        annotations_path=str(annotations_path),
    )

    print(f"Found {len(dataset.classes)} classes: {dataset.classes}")

    # Iterate through the dataset with a progress bar
    for image_path, image, detections in tqdm(dataset):
        if image is None:
            continue
        image_name = Path(image_path).stem
        image_group = image_name[0]
        # image_side = area_data[image_group]['lado_cuadrado']
        # image_resize_factor = int(resize_factor * image_side)

        # Iterate through each detection in the image
        for i, detection in enumerate(detections):
            # The detection object contains xyxy, mask, confidence, class_id, etc.
            xyxy, _, _, class_id, _, _ = detection

            # Get the class name for the current detection
            class_name = dataset.classes[class_id].lower()

            # Create a directory for the class if it doesn't exist
            class_dir = output_dir / class_name 
            class_dir.mkdir(parents=True, exist_ok=True)

            # Crop the detection from the image using its bounding box
            x1, y1, x2, y2 = map(int, xyxy)
            cropped_image = image[y1:y2, x1:x2]

            # Ensure the cropped image is not empty before saving
            if cropped_image.size == 0:
                print(f"  - Skipping empty crop for detection {i} in {image_name}.png")
                continue

            cropped_image = cv2.resize(cropped_image, (IMG_TARGET_SIDE, IMG_TARGET_SIDE))
            # x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x1, y1, x2-x1, y2-y1, image_resize_factor*image_resize_factor, image.shape[1], image.shape[0])
            # cropped_image = cv2.resize(image[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))


            # Generate a unique filename for the cropped image
            cropped_image_filename = f"{image_name}_{i}.png"
            cropped_image_path = class_dir / cropped_image_filename

            # Save the cropped image
            cv2.imwrite(str(cropped_image_path), cropped_image)

    print(f"\n✅ Processing complete. Cropped images are saved in '{output_dir}'.")


In [4]:
def remove_images_with_black_markings(directory_path: Path, black_threshold=10, white_threshold=245, percentage_threshold=1.0):
    """
    Identifies images in a directory that have a significant number of black pixels.

    Args:
        directory_path (Path): The path to the directory containing images.
        black_threshold (int): Pixel intensity value (0-255). Pixels below this
                               value are considered 'black'. Defaults to 10.
        percentage_threshold (float): The percentage of black pixels required
                                      to classify an image as having markings.
                                      (e.g., 1.0 for 1%). Defaults to 1.0.

    Returns:
        list: A list of filenames for images that have significant black markings.
    """
    images_with_markings = 0
    
    # A list of common image file extensions to check
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    print(f"Scanning directory: {directory_path}")

    images_paths = [str(path) for path in directory_path.rglob('*')]    # Iterate over every file in the directory
    for filename in tqdm(images_paths):
        # Check for a valid image extension
        if not any(filename.lower().endswith(ext) for ext in valid_extensions):
            continue

        file_path = os.path.join(directory_path, filename)

        try:
            # Read the image using OpenCV
            image = cv2.imread(file_path)

            if image is None:
                print(f"Warning: Could not read image {filename}. Skipping.")
                continue

            # Convert the image to grayscale for easier processing
            gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Calculate the total number of pixels
            total_pixels = gray_image.size

            # Count the number of pixels that are 'black' (below the threshold)
            black_pixels = np.sum(gray_image <= black_threshold)

            white_pixels = np.sum(gray_image >= white_threshold)

            # Calculate the percentage of black pixels
            percentage_of_solid_pixels = ((black_pixels + white_pixels) / total_pixels) * 100

            # If the percentage exceeds the threshold, remove image
            if percentage_of_solid_pixels > percentage_threshold:
                images_with_markings += 1
                os.remove(file_path)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print(f"Removed {images_with_markings} images with markings")



In [5]:
def process_ina_annotation(coco_path):
    """
    This function will process the COCO annotations from INA experts and give it a standard format 
    where the phases of cells will be in the cateogry_id field instead of in the attributes field.
    This is in order to process the annotations with supervision library.
    """

    # --- Validations ---
    if not os.path.isfile(coco_path):
        print(f"COCO JSON file not found: {coco_path}")
        return

    # --- Configuration ---
    # Adjust these paths to match your project structure
    output_path = os.path.dirname(coco_path)
    new_coco_filename = os.path.basename(coco_path).replace('.json', '_processed.json')
    new_coco_path = os.path.join(output_path, new_coco_filename)

    # --- Preprocessing Steps ---

    # 1. Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)

    # 2. Load the original COCO JSON data
    print(f"Loading original annotations from: {coco_path}")
    with open(coco_path, 'r') as f:
        coco_data = json.load(f)

    print("Preprocessing annotations...")

    # 3. Discover unique class names from the 'Fase' attribute and create a mapping
    # Using OrderedDict to maintain a consistent order
    class_names = sorted(list(set(
        ann.get('attributes', {}).get('Fase').replace('f','ph').lower()
        for ann in coco_data['annotations']
        if ann.get('attributes', {}).get('Fase') is not None
    )))

    # Create a mapping from class name to a new category ID (starting from 1)
    class_to_id = {name: i + 1 for i, name in enumerate(class_names)}
    print(f"\nDiscovered and mapped classes: {class_to_id}")

    # 4. Create the new categories list for the output JSON
    new_categories = [
        {"id": cat_id, "name": name, "supercategory": ""}
        for name, cat_id in class_to_id.items()
    ]

    # 5. Create a new list of annotations with updated category_id
    new_annotations = []
    for ann in coco_data['annotations']:
        attributes = ann.get('attributes', {})
        fase = attributes.get('Fase').replace('f','ph').lower()

        # Only include annotations that have a 'Fase' we are interested in
        if fase in class_to_id:
            new_ann = ann.copy()
            new_ann['category_id'] = class_to_id[fase]
            new_annotations.append(new_ann)

    # 6. Assemble the new COCO data structure
    new_coco_data = {
        'licenses': coco_data.get('licenses', []),
        'info': coco_data.get('info', {}),
        'categories': new_categories,
        'images': coco_data.get('images', []),
        'annotations': new_annotations
    }

    # 7. Save the new COCO JSON file
    print(f"\nSaving processed annotations to: {new_coco_path}")
    with open(new_coco_path, 'w') as f:
        json.dump(new_coco_data, f, indent=4)

    print("\nPreprocessing complete!")
    print(f"You can now use '{new_coco_path}' with supervision.")
    return new_coco_path



In [6]:
def remove_repeated_images(dir):
    """
    Remove repeatead images in the new datasets from roboflow since between datasets images are repeatead but with slightly different names    
    """
    def get_base_name(filename):
        for ext in ["_jpg", "_png"]:
            idx = filename.find(ext)
            if idx != -1:
                return filename[:idx+4]  # include the extension marker
        return filename

    files_by_base = defaultdict(list)
    for dirpath, dirnames, filenames in os.walk(dir):
        for filename in filenames:
            if filename.lower().endswith('.json'):
                continue  # Skip .json files
            base = get_base_name(filename)
            files_by_base[base].append(os.path.join(dirpath, filename))

    for file_list in files_by_base.values():
        for file_to_delete in file_list[1:]:
            try:
                os.remove(file_to_delete)
                print(f"Deleted: {file_to_delete}")
            except Exception as e:
                print(f"Error deleting {file_to_delete}: {e}")

In [7]:
def extract_value(annotation_str):
    """
    Define a function to extract the value from the annotations column
    """
    annotation_list = json.loads(annotation_str.lower())
    return annotation_list[0]['value'] if annotation_list and 'value' in annotation_list[0] else None

def extract_filename(subject_data_str):
    """
    Parses a JSON string from the 'subject_data' column
    and extracts the 'Filename' value from the nested dictionary.
    """
    if not isinstance(subject_data_str, str):
        return None
    try:
        # Load the string as a JSON object
        data = json.loads(subject_data_str)
        if not data:
            return None # Handles empty JSON object '{}'
            
        # The outer dictionary has a dynamic key. We get its value,
        # which is the inner dictionary.
        inner_dict = next(iter(data.values()))
        
        # Return the value of the 'Filename' key, or None if it doesn't exist
        return inner_dict.get('Filename')
    except (json.JSONDecodeError, StopIteration, AttributeError):
        # Handle cases where the string is not valid JSON,
        # or doesn't have the expected structure.
        return None

### Generate classifications from all datasets

Warning: When croping images some images won't be found since they are not tagged and the supervision library will warn about making the output dirty but it will be cropping images that are found anyway

#### Crops images from ina coco annotations

In [ ]:
IMAGES_DIR = Path(os.path.join(MEDIA_PATH, 'images', 'ina', 'tagged_images', 'input'))
COCO_ANNOTATIONS_PATH = Path(os.path.join(MEDIA_PATH, 'images', 'ina', 'tagged_images', 'corte-27-02-2024.json'))
OUTPUT_DIR = Path(os.path.join(GENERATION_DATASET_PATH, 'datasets', 'ina'))
NEW_COCO_ANNOTATIONS_PATH = Path(process_ina_annotation(COCO_ANNOTATIONS_PATH))

crop_and_save_detections(IMAGES_DIR, NEW_COCO_ANNOTATIONS_PATH, OUTPUT_DIR, '', '')

Loading original annotations from: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/corte-27-02-2024.json
Preprocessing annotations...

Discovered and mapped classes: {'anaphase': 1, 'desconocido': 2, 'interphase': 3, 'metaphase': 4, 'prophase': 5, 'telophase': 6}

Saving processed annotations to: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/corte-27-02-2024_processed.json

Preprocessing complete!
You can now use '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/corte-27-02-2024_processed.json' with supervision.
Loading dataset...
Found 6 classes: ['anaphase', 'desconocido', 'interphase', 'metaphase', 'prophase', 'telophase']


  0%|          | 0/393 [00:00<?, ?it/s][ WARN:0@134.554] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00001.jpg'): can't open/read file: check file path/integrity
[ WARN:0@134.554] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00002.jpg'): can't open/read file: check file path/integrity
[ WARN:0@134.554] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00003.jpg'): can't open/read file: check file path/integrity
[ WARN:0@134.554] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00004.jpg'): can't open/read file: check file path/integrity
[ WARN:0@134.554] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/im


✅ Processing complete. Cropped images are saved in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/generation/train/tagged/ina'.


#### Crops images from onion_cell_merged coco annotations

In [ ]:
IMAGES_DIR = os.path.join(MEDIA_PATH, 'images','onion_cell_merged', 'images')
OUTPUT_DIR = os.path.join(GENERATION_DATASET_PATH,  'datasets', 'onion_cell_merged')
ANNOTATIONS = {'train': 'annotations_coco_train.json','valid': 'annotations_coco_valid.json','test': 'annotations_coco_test.json'}
BLACK_PIXEL_INTENSITY_THRESHOLD = 10  # How dark a pixel must be to be 'black' (0-255)
WHITE_PIXEL_INTENSITY_THRESHOLD = 245 # How brigth a pixel must be to be 'white' (0-255)
BLACK_PIXEL_PERCENTAGE = 15.0    # What percentage of the image needs to be black or white

# with open(JSON_PATH, 'r') as f: #json with the information of the filename of the images
#     area_data = json.load(f)
# resize_factor = IMG_TARGET_SIDE/area_data['INA']['lado_cuadrado']

for split in ANNOTATIONS.keys():
    crop_and_save_detections(
        images_dir=Path(os.path.join(IMAGES_DIR, split)),
        annotations_path=Path(os.path.join(IMAGES_DIR, ANNOTATIONS[split])),
        output_dir=Path(OUTPUT_DIR),
        resize_factor='',#resize_factor,
        area_data = ''#area_data
    )

# Remove images with black or white markings
marked_images = remove_images_with_black_markings(
    Path(OUTPUT_DIR),
    black_threshold=BLACK_PIXEL_INTENSITY_THRESHOLD,
    white_threshold=WHITE_PIXEL_INTENSITY_THRESHOLD,
    percentage_threshold=BLACK_PIXEL_PERCENTAGE
)

#### Crops images from new roboflow datasets coco annotations

In [ ]:
ROBOFLOW_DATASETS = ['Mitosis.v1i.coco', 'Mitosis.v15-5.coco', 'Mitosis Counter.v7i.coco', 'mitosis_baseline.v1i.coco']
IMAGES_DIR = os.path.join(MEDIA_PATH, 'nuevos datasets')
OUTPUT_DIR = os.path.join(GENERATION_DATASET_PATH,  'datasets', 'roboflow')
ANNOTATIONS = {'train': 'annotations_coco_train.json','valid': 'annotations_coco_valid.json','test': 'annotations_coco_test.json'}

remove_repeated_images(IMAGES_DIR)

for dataset in ROBOFLOW_DATASETS:
    images_dir = os.path.join(IMAGES_DIR, dataset)
    for split in ANNOTATIONS.keys():
        crop_and_save_detections(
            images_dir=Path(os.path.join(images_dir, split)),
            annotations_path=Path(os.path.join(images_dir, ANNOTATIONS[split])),
            output_dir=Path(OUTPUT_DIR),
            resize_factor='',#resize_factor,
            area_data = ''#area_data
        )

#### Separate images tagged from zooniverse

In [ ]:
import glob
import pandas as pd
import shutil
import os
from tqdm import tqdm

# Configuration
CSV_FILE_PATH = "/home/nicolas/Descargas/automating-allium-cepa-assay-analysis-with-ai-classifications.csv"
INA_CROPS_PATH = os.path.join(CROPPED_PATH, 'ina', 'images')
RB_CROPS_PATH = os.path.join(CROPPED_PATH, 'onion_cell_merged', 'images')
INA_CROPS = glob.glob(os.path.join(INA_CROPS_PATH, '*'))
RB_CROPS = glob.glob(os.path.join(RB_CROPS_PATH, '*/*'))

ALL_CROPS = INA_CROPS + RB_CROPS

# Build a fast lookup dictionary: {basename: full_path}
all_crops_dict = {os.path.basename(path).lower(): path for path in ALL_CROPS}

# Load data
df = pd.read_csv(CSV_FILE_PATH)

# Process data
annotations_subject_data = df[['annotations', 'subject_data']].copy()
annotations_subject_data['stage'] = annotations_subject_data['annotations'].apply(extract_value).str.lower()
annotations_subject_data['filename'] = annotations_subject_data['subject_data'].apply(extract_filename)
results = annotations_subject_data[['stage', 'filename']].dropna()

# Clean results: get most common stage per filename
cleaned_results = results.groupby('filename')['stage'].agg(lambda x: x.mode()[0]).reset_index()
stage_to_filenames_dict = cleaned_results.groupby('stage')['filename'].apply(list).to_dict()

# Copy images to class folders
for stage, filenames in stage_to_filenames_dict.items():
    output_path = os.path.join(GENERATION_DATASET_PATH,  'datasets', 'zooniverse', stage)
    os.makedirs(output_path, exist_ok=True)
    for image in tqdm(filenames, desc=f"Copying for stage {stage}"):
        src = all_crops_dict.get(image)
        dst = os.path.join(output_path, image)
        if src and os.path.exists(src):
            shutil.copy2(src, dst)
        else:
            print(f"Source file {src} does not exist. Skipping.")

### Manual task: control images in classes and create following structure

### The intended directory structure is the following:

```text
|-- classification/
    |-- train/
        |-- tagged/
            |-- prophase/
                |-- example_prophase_image.png
            |-- metaphase/
                |-- example_metaphase_image.png
            |-- anaphase/
                |-- example_anaphase_image.png
            |-- telophase/
                |-- example_telophase_image.png
        |-- untagged/
            |-- example_untagged_image.png
```

In [ ]:
raise ValueError("Please organize images in the classes folders") #Intended to stop the script here to mannualy organize classes

ValueError: Please organize images in the classes folders

### Creation of train - test split

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

def split_dataset(source_dir, test_dir, test_size=0.2, seed=SEED):
    os.makedirs(test_dir, exist_ok=True)
    classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    for class_name in classes:
        class_path = os.path.join(source_dir, class_name)
        images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
        train_imgs, test_imgs = train_test_split(images, test_size=test_size, random_state=seed)
        # Create class subfolder in test_dir
        test_class_dir = os.path.join(test_dir, class_name)
        os.makedirs(test_class_dir, exist_ok=True)
        # Copy test images
        for img in test_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(test_class_dir, img)
            shutil.copy2(src, dst)
        print(f"Class '{class_name}': {len(test_imgs)} test images copied.")

TRAIN_PATH = os.path.join(GENERATION_DATASET_PATH,  'train', 'tagged')
TEST_PATH = os.path.join(GENERATION_DATASET_PATH,  'test')
split_dataset(TRAIN_PATH, TEST_PATH, test_size=0.2)


Class 'metaphase': 56 test images copied.
Class 'telophase': 52 test images copied.
Class 'anaphase': 45 test images copied.
Class 'prophase': 89 test images copied.


### Pending manual task

After the classes directories are created it is needed to augment the images. For that there is a notebook (data_augmentation.ipynb) which will have to be run for the classes and the untagged images three times for each. The prophase folder can be augmented only once since it is the class with the most amount of images

### Unused code

In [ ]:
from tqdm import tqdm

"""
Codigo para renombrar las imagenes taggeadas para que tengan el mismo nombre que en el json
"""

IMAGES_DIR = (os.path.join(MEDIA_PATH, "images", "ina", "tagged_images", 'input'))
ANNOTATIONS_PATH = (os.path.join(MEDIA_PATH, 'images', 'ina', 'tagged_images', 'corte-27-02-2024.json'))
OUTPUT_DIR = (os.path.join(MEDIA_PATH, "images", "ina", "tagged_images"))

images_paths = [IMAGES_DIR + '/' + item for item in sorted(os.listdir(IMAGES_DIR))]

with open(ANNOTATIONS_PATH, 'r') as f: #json with the information of the filename of the images
    data = json.load(f)

for image in tqdm(images_paths):
    image_id = os.path.basename(image).split('.')[0]

    if image_id.isnumeric() == False:
        print('Ignoring non-numeric image:', image)
        continue

    value = data['images'][int(image_id) - 1]['file_name']
    os.rename(image, f'{IMAGES_DIR}/{value}')

In [ ]:
import cv2
import os
from tqdm import tqdm

def resize_images_in_directory(dataset_path, output_dir, target_size=(200, 200)):
    """
    Searches for images starting with 'IMG' in a directory, resizes them,
    and saves them to an output directory.

    Args:
        dataset_path (str): The path to the directory containing the images.
        output_dir (str): The path to the directory where resized images will be saved.
        target_size (tuple): A tuple (width, height) for the resized images.
    """
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # Check if the dataset path exists
    if not os.path.isdir(dataset_path):
        print(f"Error: The directory '{dataset_path}' was not found.")
        return

    print(f"Searching for images in '{dataset_path}'...")

    # Loop through all the files in the source directory
    for filename in tqdm(os.listdir(dataset_path)):
        # Check if the file name starts with 'IMG' and is a supported image format
        if filename.startswith('IMG') and filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_filepath = os.path.join(dataset_path, filename)

            # Read the image using OpenCV
            image = cv2.imread(input_filepath)

            # Check if the image was loaded successfully
            if image is None:
                print(f"Warning: Could not read image {input_filepath}. Skipping.")
                continue

            # Resize the image
            # cv2.INTER_AREA is generally good for shrinking images.
            resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

            # Construct the output file path
            output_filepath = os.path.join(output_dir, filename)

            # Save the resized image
            cv2.imwrite(output_filepath, resized_image)
            print(f"Resized and saved '{filename}' to '{output_filepath}'")

    print("\nProcessing complete.")

DATASET_PATH = os.path.join(CROPPED_PATH, 'd') # Run again for 'not' dataset
resize_images_in_directory(DATASET_PATH, DATASET_PATH)

